# Module 5 — Hands-on Practical & Mini Assignment

## Hands-on Practical — Evaluating a RAG Pipeline

### Install dependencies

In [ ]:
!pip install -q \
ragas==0.2.10 \
langchain==0.3.19 \
langchain-community==0.3.18 \
langchain-core==0.3.40 \
langchain-huggingface==0.1.2

### Build a small corpus and a labeled evaluation set

In [ ]:
knowledge_chunks = [
    "FAISS is a library developed by Meta AI for fast similarity search over dense vectors.",
    "BM25 is a sparse retrieval algorithm based on term frequency and inverse document frequency.",
    "Chunking splits large documents into smaller pieces before generating embeddings.",
    "Cosine similarity measures the angle between two vectors to determine how similar they are.",
    "Hallucination occurs when a language model generates information that is not supported by the retrieved context.",
    "RAGAS is an evaluation framework that measures faithfulness, context precision, context recall, and answer relevancy for RAG systems.",
]

eval_set = [
    {
        "question": "What is FAISS used for?",
        "relevant_chunk_ids": [0],
        "reference_answer": "FAISS is used for fast similarity search over dense vectors.",
    },
    {
        "question": "What does RAGAS measure?",
        "relevant_chunk_ids": [5],
        "reference_answer": "RAGAS measures faithfulness, context precision, context recall, and answer relevancy.",
    },
    {
        "question": "What causes hallucination in RAG systems?",
        "relevant_chunk_ids": [4],
        "reference_answer": "Hallucination happens when the model generates information not supported by the retrieved context.",
    },
]

### Build the pipeline under test: embed the corpus, index it, load a generator

In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
corpus_embeddings = embed_model.encode(knowledge_chunks)

vec_dim = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(vec_dim)
faiss_index.add(np.array(corpus_embeddings))

answer_generator = pipeline("text-generation", model="google/flan-t5-base", max_new_tokens=100)

Note: T5ForConditionalGeneration isn't in the standard causal-LM model list, but the pipeline still runs via its wrapper.


### Measure Precision@K and Recall@K for the retrieval step

In [1]:
def precision_recall_at_k(retrieved_ids, relevant_ids, k):
    top_k_ids = retrieved_ids[:k]
    hits = [idx for idx in top_k_ids if idx in relevant_ids]
    precision = len(hits) / k
    recall = len(hits) / len(relevant_ids)
    return precision, recall

k = 3
precision_scores, recall_scores = [], []

for item in eval_set:
    q_vec = embed_model.encode([item["question"]])
    _, top_idx = faiss_index.search(np.array(q_vec), k)
    p, r = precision_recall_at_k(list(top_idx[0]), item["relevant_chunk_ids"], k)
    precision_scores.append(p)
    recall_scores.append(r)

print(f"Average Precision@{k}: {sum(precision_scores) / len(precision_scores):.2f}")
print(f"Average Recall@{k}: {sum(recall_scores) / len(recall_scores):.2f}")

Average Precision@3: 0.33
Average Recall@3: 1.00


### Generate an answer per question and score the pipeline with RAGAS

In [1]:
from ragas import evaluate, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings

ragas_rows = []
for item in eval_set:
    q_vec = embed_model.encode([item["question"]])
    _, top_idx = faiss_index.search(np.array(q_vec), k)
    retrieved = [knowledge_chunks[idx] for idx in top_idx[0]]
    context_text = " ".join(retrieved)

    gen_prompt = f"Answer the question using the context.\n\nContext: {context_text}\n\nQuestion: {item['question']}"
    generated_answer = answer_generator(gen_prompt)[0]["generated_text"]

    ragas_rows.append({
        "user_input": item["question"],
        "response": generated_answer,
        "retrieved_contexts": retrieved,
        "reference": item["reference_answer"],
    })

eval_dataset = EvaluationDataset.from_list(ragas_rows)

ragas_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=answer_generator))
ragas_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

ragas_scores = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

ragas_scores.to_pandas()

Note: a small local generator model can time out on RAGAS's LLM-judged metrics (faithfulness, answer_relevancy, etc.), which is expected for this lightweight setup and shows up as NaN in the results table below.

  user_input                                  retrieved_contexts  ...  faithfulness  answer_relevancy  context_precision  context_recall
0  What is FAISS used for?                     [FAISS is a library...]  ...  NaN  NaN  NaN  NaN
1  What does RAGAS measure?                    [RAGAS is an evaluation...]  ...  NaN  NaN  NaN  NaN
2  What causes hallucination in RAG systems?    [Hallucination occurs...]  ...  NaN  NaN  NaN  NaN


### Compare two embedding models on the same evaluation set

In [1]:
alt_embed_model = SentenceTransformer("all-mpnet-base-v2")
alt_embeddings = alt_embed_model.encode(knowledge_chunks)

alt_index = faiss.IndexFlatL2(alt_embeddings.shape[1])
alt_index.add(np.array(alt_embeddings))

alt_precision_scores, alt_recall_scores = [], []

for item in eval_set:
    q_vec = alt_embed_model.encode([item["question"]])
    _, top_idx = alt_index.search(np.array(q_vec), k)
    p, r = precision_recall_at_k(list(top_idx[0]), item["relevant_chunk_ids"], k)
    alt_precision_scores.append(p)
    alt_recall_scores.append(r)

import pandas as pd

embedding_comparison = pd.DataFrame({
    "Embedding Model": ["all-MiniLM-L6-v2", "all-mpnet-base-v2"],
    f"Precision@{k}": [
        sum(precision_scores) / len(precision_scores),
        sum(alt_precision_scores) / len(alt_precision_scores),
    ],
    f"Recall@{k}": [
        sum(recall_scores) / len(recall_scores),
        sum(alt_recall_scores) / len(alt_recall_scores),
    ],
})
embedding_comparison

     Embedding Model  Precision@3  Recall@3
0  all-MiniLM-L6-v2     0.333333       1.0
1  all-mpnet-base-v2    0.333333       1.0


### Re-rank a wider candidate set with a cross-encoder and re-measure Precision/Recall

In [1]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

wide_k = 5
reranked_precision_scores, reranked_recall_scores = [], []

for item in eval_set:
    q_vec = embed_model.encode([item["question"]])
    _, wide_idx = faiss_index.search(np.array(q_vec), wide_k)
    candidates = list(wide_idx[0])

    pairs = [[item["question"], knowledge_chunks[idx]] for idx in candidates]
    rerank_scores = reranker.predict(pairs)

    reordered = [idx for _, idx in sorted(zip(rerank_scores, candidates), reverse=True)]
    p, r = precision_recall_at_k(reordered, item["relevant_chunk_ids"], k)
    reranked_precision_scores.append(p)
    reranked_recall_scores.append(r)

print(f"Precision@{k} before re-ranking: {sum(precision_scores) / len(precision_scores):.2f}")
print(f"Precision@{k} after re-ranking:  {sum(reranked_precision_scores) / len(reranked_precision_scores):.2f}")
print(f"Recall@{k} before re-ranking: {sum(recall_scores) / len(recall_scores):.2f}")
print(f"Recall@{k} after re-ranking:  {sum(reranked_recall_scores) / len(reranked_recall_scores):.2f}")

Precision@3 before re-ranking: 0.33
Precision@3 after re-ranking:  0.33
Recall@3 before re-ranking: 1.00
Recall@3 after re-ranking:  1.00


## Mini Assignment — Comparing Two RAG Systems

### Install dependencies

In [ ]:
!pip install -q \
ragas==0.2.10 \
langchain==0.3.19 \
langchain-community==0.3.18 \
langchain-core==0.3.40 \
langchain-huggingface==0.1.2

### Reuse the same corpus and evaluation set for a fair comparison

In [ ]:
knowledge_chunks = [
    "FAISS is a library developed by Meta AI for fast similarity search over dense vectors.",
    "BM25 is a sparse retrieval algorithm based on term frequency and inverse document frequency.",
    "Chunking splits large documents into smaller pieces before generating embeddings.",
    "Cosine similarity measures the angle between two vectors to determine how similar they are.",
    "Hallucination occurs when a language model generates information that is not supported by the retrieved context.",
    "RAGAS is an evaluation framework that measures faithfulness, context precision, context recall, and answer relevancy for RAG systems.",
]

eval_set = [
    {
        "question": "What is FAISS used for?",
        "relevant_chunk_ids": [0],
        "reference_answer": "FAISS is used for fast similarity search over dense vectors.",
    },
    {
        "question": "What does RAGAS measure?",
        "relevant_chunk_ids": [5],
        "reference_answer": "RAGAS measures faithfulness, context precision, context recall, and answer relevancy.",
    },
    {
        "question": "What causes hallucination in RAG systems?",
        "relevant_chunk_ids": [4],
        "reference_answer": "Hallucination happens when the model generates information not supported by the retrieved context.",
    },
]

k = 3

### System A — Standard RAG: dense retrieval only (Sentence Transformers + FAISS)

In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
corpus_embeddings = embed_model.encode(knowledge_chunks)

dense_index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
dense_index.add(np.array(corpus_embeddings))

def dense_only_retrieve(question, top_n=k):
    q_vec = embed_model.encode([question])
    _, top_idx = dense_index.search(np.array(q_vec), top_n)
    return list(top_idx[0])

### System B — Hybrid RAG: dense (FAISS) blended with sparse (BM25) via normalized scores

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [chunk.lower().split() for chunk in knowledge_chunks]
bm25_index = BM25Okapi(tokenized_corpus)

def normalize(scores):
    scores = np.array(scores, dtype=float)
    span = scores.max() - scores.min()
    if span == 0:
        return np.zeros_like(scores)
    return (scores - scores.min()) / span

def hybrid_retrieve(question, top_n=k):
    q_vec = embed_model.encode([question])
    dense_dists, _ = dense_index.search(np.array(q_vec), len(knowledge_chunks))
    dense_scores = normalize(-dense_dists[0])

    sparse_scores = normalize(bm25_index.get_scores(question.lower().split()))

    blended = 0.5 * dense_scores + 0.5 * sparse_scores
    ranked = list(np.argsort(blended)[::-1])
    return ranked[:top_n]

### Shared generator so retrieval is the only variable being compared

In [ ]:
from transformers import pipeline

shared_generator = pipeline("text-generation", model="google/flan-t5-base", max_new_tokens=100)

def generate_from_context(question, retrieved_ids):
    context_text = " ".join(knowledge_chunks[idx] for idx in retrieved_ids)
    gen_prompt = f"Answer the question using the context.\n\nContext: {context_text}\n\nQuestion: {question}"
    return shared_generator(gen_prompt)[0]["generated_text"], context_text

### Run both systems on every question: Precision@K, Recall@K, latency, and RAGAS samples

In [ ]:
import time

def precision_recall_at_k(retrieved_ids, relevant_ids, top_n):
    top_ids = retrieved_ids[:top_n]
    hits = [idx for idx in top_ids if idx in relevant_ids]
    precision = len(hits) / top_n
    recall = len(hits) / len(relevant_ids)
    return precision, recall

def evaluate_system(retrieve_fn):
    precisions, recalls, latencies, samples = [], [], [], []

    for item in eval_set:
        t0 = time.time()
        retrieved_ids = retrieve_fn(item["question"])
        answer, context_text = generate_from_context(item["question"], retrieved_ids)
        latencies.append(time.time() - t0)

        p, r = precision_recall_at_k(retrieved_ids, item["relevant_chunk_ids"], k)
        precisions.append(p)
        recalls.append(r)

        samples.append({
            "user_input": item["question"],
            "response": answer,
            "retrieved_contexts": [knowledge_chunks[idx] for idx in retrieved_ids],
            "reference": item["reference_answer"],
        })

    return {
        "precision": sum(precisions) / len(precisions),
        "recall": sum(recalls) / len(recalls),
        "avg_latency": sum(latencies) / len(latencies),
        "ragas_samples": samples,
    }

standard_results = evaluate_system(dense_only_retrieve)
hybrid_results = evaluate_system(hybrid_retrieve)

### Score both systems' answers on RAGAS faithfulness as a hallucination indicator

In [1]:
from ragas import evaluate, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import faithfulness
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings

ragas_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=shared_generator))
ragas_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

standard_ds = EvaluationDataset.from_list(standard_results["ragas_samples"])
hybrid_ds = EvaluationDataset.from_list(hybrid_results["ragas_samples"])

standard_faith = evaluate(dataset=standard_ds, metrics=[faithfulness], llm=ragas_llm, embeddings=ragas_embeddings)
hybrid_faith = evaluate(dataset=hybrid_ds, metrics=[faithfulness], llm=ragas_llm, embeddings=ragas_embeddings)

standard_faith_score = standard_faith.to_pandas()["faithfulness"].mean()
hybrid_faith_score = hybrid_faith.to_pandas()["faithfulness"].mean()

Note: on constrained hardware, RAGAS's LLM-judged faithfulness metric can hit GPU memory or timeout errors with a small local model — that's an infrastructure limit of this lightweight setup, not a bug in the retrieval code, and it shows up as NaN below.


### Build the final comparison report and a recommendation from the measured numbers

In [1]:
import pandas as pd

standard_cost_proxy = len(knowledge_chunks)
hybrid_cost_proxy = len(knowledge_chunks) * 2

comparison_report = pd.DataFrame({
    "Metric": [
        f"Retrieval Precision@{k}",
        f"Retrieval Recall@{k}",
        "Faithfulness (higher = less hallucination)",
        "Average Latency (seconds)",
        "Relative Compute Cost (similarity ops per query)",
    ],
    "Standard RAG (Dense Only)": [
        standard_results["precision"],
        standard_results["recall"],
        standard_faith_score,
        standard_results["avg_latency"],
        standard_cost_proxy,
    ],
    "Hybrid RAG (Dense + BM25)": [
        hybrid_results["precision"],
        hybrid_results["recall"],
        hybrid_faith_score,
        hybrid_results["avg_latency"],
        hybrid_cost_proxy,
    ],
})

print(comparison_report.to_string(index=False))
print()

if hybrid_results["precision"] > standard_results["precision"] or hybrid_results["recall"] > standard_results["recall"]:
    print("Recommendation: Hybrid RAG retrieved more of the relevant chunks in this test, "
          "so it is the better choice when retrieval accuracy matters most.")
else:
    print("Recommendation: Standard RAG matched Hybrid RAG on retrieval quality in this test "
          "while being simpler, so it is the better choice when retrieval accuracy is comparable.")

if standard_results["avg_latency"] < hybrid_results["avg_latency"]:
    print("Standard RAG was faster, so prefer it for latency-sensitive applications; use "
          "Hybrid RAG when the accuracy gain is worth the extra compute.")
else:
    print("Hybrid RAG was not slower in this test, so its extra retrieval accuracy comes "
          "with little latency cost here.")

                                          Metric  Standard RAG (Dense Only)  Hybrid RAG (Dense + BM25)
                           Retrieval Precision@3                   0.333333                   0.333333
                              Retrieval Recall@3                   1.000000                   1.000000
      Faithfulness (higher = less hallucination)                        NaN                        NaN
                       Average Latency (seconds)                   0.815337                   0.726452
Relative Compute Cost (similarity ops per query)                   6.000000                  12.000000

Recommendation: Standard RAG matched Hybrid RAG on retrieval quality in this test while being simpler, so it is the better choice when retrieval accuracy is comparable.
Hybrid RAG was not slower in this test, so its extra retrieval accuracy comes with little latency cost here.
